# VRP Strategy Research — Put Write & Skew Trading

**Purpose**: Offline exploration of two additional Volatility Risk Premium strategies
using only VIX as the IV proxy (no real options chain data needed).

**Strategies explored**:
1. **Put Write** — Sell 30-day 5% OTM puts on SPY monthly.  Simpler than straddle,
   captures equity put skew premium.
2. **Skew Trading** — Sell OTM put / buy ATM put (vertical spread) when put skew
   is steep.  Isolates the skew premium with capped tail risk.

**Baseline**: Delta Neutral VRP straddle (already live paper trading).

---

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.stats import norm
from datetime import date, timedelta
import warnings
warnings.filterwarnings('ignore')

# Plotting
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

## 1. Load SPY and VIX Data (2018–2024)

In [ ]:
# Download SPY and VIX
spy = yf.download('SPY', start='2018-01-01', end='2024-12-31', auto_adjust=True)
vix = yf.download('^VIX', start='2018-01-01', end='2024-12-31', auto_adjust=True)

# Align on common dates
spy_close = spy['Close'].squeeze().dropna()
vix_close = vix['Close'].squeeze().dropna()
common_idx = spy_close.index.intersection(vix_close.index)
spy_close = spy_close.loc[common_idx]
vix_close = vix_close.loc[common_idx]

print(f'Date range: {common_idx[0].date()} to {common_idx[-1].date()}')
print(f'Trading days: {len(common_idx)}')
print(f'SPY range: ${spy_close.min():.2f} – ${spy_close.max():.2f}')
print(f'VIX range: {vix_close.min():.2f} – {vix_close.max():.2f}')

## 2. Black-Scholes Helpers

In [ ]:
def bsm_put(S, K, T, r, sigma):
    """Black-Scholes put price."""
    if T < 1e-10 or sigma < 1e-10:
        return max(K - S, 0.0)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return float(K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1))


def bsm_call(S, K, T, r, sigma):
    """Black-Scholes call price."""
    if T < 1e-10 or sigma < 1e-10:
        return max(S - K, 0.0)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return float(S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2))


def bsm_put_delta(S, K, T, r, sigma):
    """BSM put delta (negative for long put)."""
    if T < 1e-10 or sigma < 1e-10:
        return -1.0 if S < K else 0.0
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return float(norm.cdf(d1) - 1.0)


print('BSM helpers loaded.')

## 3. Strategy 1 — Put Write

**Rules**:
- Each month (21 trading days), sell a 30-day 5% OTM put on SPY.
- Strike = 0.95 × spot.  IV = VIX / 100.
- Credit = BSM put price.
- Hold to expiration.  P&L = credit − intrinsic at expiry.
- Position sizing: risk 2% of equity per trade.  Max loss = strike (gap to zero),
  but we use 3× credit as practical worst-case.
- Commission: $0.65 per contract.

In [ ]:
def run_put_write_backtest(
    spy_close: pd.Series,
    vix_close: pd.Series,
    initial_capital: float = 1_000_000,
    otm_pct: float = 0.05,
    dte: int = 30,
    risk_per_trade_pct: float = 0.02,
    max_loss_mult: float = 3.0,
    entry_interval: int = 21,
    profit_target_pct: float = 0.50,
    time_exit_dte: int = 7,
    commission: float = 0.65,
    r: float = 0.04,
) -> tuple[list[dict], list[tuple], pd.Series]:
    """Run a theoretical put-write backtest.

    Returns (trades, equity_curve, equity_series).
    """
    trades: list[dict] = []
    equity_curve: list[tuple] = []
    equity = initial_capital
    last_entry_idx = -entry_interval  # allow first entry immediately

    dates = spy_close.index
    n = len(dates)
    active_trade: dict | None = None

    for i in range(n):
        S = float(spy_close.iloc[i])
        vix_val = float(vix_close.iloc[i])
        sigma = vix_val / 100.0
        dt = dates[i]

        # --- Manage active trade ---
        if active_trade is not None:
            days_held = i - active_trade['entry_idx']
            remaining_dte = active_trade['dte'] - days_held

            if remaining_dte <= 0:
                # Expiration
                intrinsic = max(active_trade['strike'] - S, 0)
                pnl = (active_trade['credit'] - intrinsic) * 100 - commission
                active_trade['pnl'] = pnl
                active_trade['exit_date'] = dt
                active_trade['exit_price'] = S
                active_trade['exit_reason'] = 'expiration'
                equity += pnl
                trades.append(active_trade)
                active_trade = None

            elif remaining_dte <= time_exit_dte:
                # Time exit: close at current BSM price
                T_rem = max(remaining_dte / 252, 1/252)
                current_put = bsm_put(S, active_trade['strike'], T_rem, r, sigma)
                pnl = (active_trade['credit'] - current_put) * 100 - commission * 2
                active_trade['pnl'] = pnl
                active_trade['exit_date'] = dt
                active_trade['exit_price'] = S
                active_trade['exit_reason'] = 'time_exit'
                equity += pnl
                trades.append(active_trade)
                active_trade = None

            else:
                # Check profit target
                T_rem = max(remaining_dte / 252, 1/252)
                current_put = bsm_put(S, active_trade['strike'], T_rem, r, sigma)
                unrealized = (active_trade['credit'] - current_put) * 100
                max_profit = active_trade['credit'] * 100
                if unrealized >= profit_target_pct * max_profit:
                    pnl = unrealized - commission * 2
                    active_trade['pnl'] = pnl
                    active_trade['exit_date'] = dt
                    active_trade['exit_price'] = S
                    active_trade['exit_reason'] = 'profit_target'
                    equity += pnl
                    trades.append(active_trade)
                    active_trade = None

        # --- Entry logic ---
        if active_trade is None and (i - last_entry_idx) >= entry_interval:
            K = S * (1 - otm_pct)
            T = dte / 252
            credit = bsm_put(S, K, T, r, sigma)

            if credit > 0.05:  # minimum credit
                # Position sizing
                risk_amount = equity * risk_per_trade_pct
                max_loss_per_contract = max_loss_mult * credit * 100
                contracts = max(1, int(risk_amount / max_loss_per_contract))

                active_trade = {
                    'entry_date': dt,
                    'entry_idx': i,
                    'entry_price': S,
                    'strike': K,
                    'credit': credit,
                    'contracts': contracts,
                    'dte': dte,
                    'iv': sigma,
                    'vix': vix_val,
                }
                last_entry_idx = i

        equity_curve.append((dt, equity))

    # Close any remaining trade at end
    if active_trade is not None:
        S = float(spy_close.iloc[-1])
        sigma = float(vix_close.iloc[-1]) / 100
        days_held = n - 1 - active_trade['entry_idx']
        remaining_dte = active_trade['dte'] - days_held
        T_rem = max(remaining_dte / 252, 1/252)
        current_put = bsm_put(S, active_trade['strike'], T_rem, r, sigma)
        pnl = (active_trade['credit'] - current_put) * 100 - commission * 2
        active_trade['pnl'] = pnl
        active_trade['exit_date'] = dates[-1]
        active_trade['exit_price'] = S
        active_trade['exit_reason'] = 'end_of_backtest'
        equity += pnl
        trades.append(active_trade)
        equity_curve.append((dates[-1], equity))

    eq_series = pd.Series(
        [v for _, v in equity_curve],
        index=[d for d, _ in equity_curve],
    )
    return trades, equity_curve, eq_series


print('Put write backtest engine defined.')

In [ ]:
# Run put write backtest
pw_trades, pw_eq_curve, pw_equity = run_put_write_backtest(spy_close, vix_close)

pw_returns = pw_equity.pct_change().dropna()
pw_sharpe = float(pw_returns.mean() / pw_returns.std() * np.sqrt(252)) if pw_returns.std() > 0 else 0
pw_peak = pw_equity.cummax()
pw_dd = ((pw_equity - pw_peak) / pw_peak * 100)
pw_max_dd = float(pw_dd.min())

pw_wins = sum(1 for t in pw_trades if t.get('pnl', 0) > 0)
pw_win_rate = pw_wins / len(pw_trades) * 100 if pw_trades else 0
pw_total_pnl = sum(t.get('pnl', 0) for t in pw_trades)
pw_pos_pnl = sum(t['pnl'] for t in pw_trades if t['pnl'] > 0)
pw_neg_pnl = abs(sum(t['pnl'] for t in pw_trades if t['pnl'] <= 0))
pw_pf = pw_pos_pnl / pw_neg_pnl if pw_neg_pnl > 0 else float('inf')

print('=== Put Write Strategy Results ===')
print(f'Trades:          {len(pw_trades)}')
print(f'Win rate:        {pw_win_rate:.1f}%')
print(f'Total P&L:       ${pw_total_pnl:,.0f}')
print(f'Profit factor:   {pw_pf:.2f}')
print(f'Sharpe ratio:    {pw_sharpe:.2f}')
print(f'Max drawdown:    {pw_max_dd:.2f}%')
print(f'Final equity:    ${pw_equity.iloc[-1]:,.0f}')
print(f'Annual return:   {(pw_equity.iloc[-1]/1_000_000)**(1/7) - 1:.2%}')

# Close reasons
reasons = {}
for t in pw_trades:
    r = t.get('exit_reason', 'unknown')
    reasons[r] = reasons.get(r, 0) + 1
print(f'\nClose reasons:  {reasons}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Equity curve
axes[0, 0].plot(pw_equity.index, pw_equity.values / 1e6, 'b-', linewidth=1)
axes[0, 0].set_title('Put Write — Equity Curve')
axes[0, 0].set_ylabel('Equity ($M)')
axes[0, 0].axhline(1.0, color='gray', linestyle='--', alpha=0.5)

# Drawdown
axes[0, 1].fill_between(pw_dd.index, pw_dd.values, 0, alpha=0.4, color='red')
axes[0, 1].set_title('Put Write — Drawdown')
axes[0, 1].set_ylabel('Drawdown (%)')

# Trade P&L distribution
pnls = [t['pnl'] for t in pw_trades]
axes[1, 0].hist(pnls, bins=30, edgecolor='black', alpha=0.7)
axes[1, 0].axvline(0, color='red', linestyle='--')
axes[1, 0].set_title('Put Write — Trade P&L Distribution')
axes[1, 0].set_xlabel('P&L ($)')

# P&L by VIX regime
vix_at_entry = [t['vix'] for t in pw_trades]
pnl_by_vix = [(t['vix'], t['pnl']) for t in pw_trades]
vix_arr, pnl_arr = zip(*pnl_by_vix)
colors = ['green' if p > 0 else 'red' for p in pnl_arr]
axes[1, 1].scatter(vix_arr, pnl_arr, c=colors, alpha=0.6, s=30)
axes[1, 1].axhline(0, color='black', linestyle='-', alpha=0.3)
axes[1, 1].set_title('Put Write — P&L vs VIX at Entry')
axes[1, 1].set_xlabel('VIX')
axes[1, 1].set_ylabel('P&L ($)')

plt.tight_layout()
plt.savefig('../reports/put_write_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to reports/put_write_analysis.png')

## 4. Strategy 2 — Skew Trading (Put Vertical Spread)

**Rules**:
- Measure put skew: OTM put IV − ATM IV.  Use a simple skew model:
  - VIX < 20 → skew ≈ 2%
  - VIX 20–30 → skew ≈ 4%
  - VIX > 30 → skew ≈ 6%
- Enter when skew > 3%: sell 5% OTM put, buy ATM put (vertical spread).
- Credit = OTM put premium − ATM put premium.
- Exit at expiration or when skew normalizes below 2%.
- Position sizing: 2% equity risk per trade.

In [ ]:
def estimate_skew(vix: float) -> float:
    """Estimate put skew (OTM IV − ATM IV) from VIX level.

    Based on typical SPX put skew behaviour:
    - Low vol (VIX < 15): skew ~1.5%
    - Normal vol (15–20): skew ~2.5%
    - Elevated vol (20–30): skew ~4%
    - High vol (30+): skew ~6%
    """
    if vix < 15:
        return 0.015
    elif vix < 20:
        return 0.015 + (vix - 15) / 5 * 0.010  # linear ramp 1.5% → 2.5%
    elif vix < 30:
        return 0.025 + (vix - 20) / 10 * 0.015  # linear ramp 2.5% → 4%
    else:
        return 0.04 + (vix - 30) / 10 * 0.02    # linear ramp 4% → 6%


def run_skew_trading_backtest(
    spy_close: pd.Series,
    vix_close: pd.Series,
    initial_capital: float = 1_000_000,
    otm_pct: float = 0.05,
    dte: int = 30,
    risk_per_trade_pct: float = 0.02,
    max_loss_mult: float = 2.0,  # tighter because spread is defined-risk
    skew_entry: float = 0.03,
    skew_exit: float = 0.02,
    profit_target_pct: float = 0.50,
    time_exit_dte: int = 7,
    commission: float = 0.65,
    r: float = 0.04,
) -> tuple[list[dict], list[tuple], pd.Series]:
    """Run a theoretical skew-trading backtest.

    The skew spread: short 5% OTM put (higher IV) + long ATM put (lower IV).
    Net credit = OTM_put - ATM_put.  Max loss = width - credit.
    """
    trades: list[dict] = []
    equity_curve: list[tuple] = []
    equity = initial_capital
    last_entry_idx = -22  # allow first entry

    dates = spy_close.index
    n = len(dates)
    active_trade: dict | None = None

    for i in range(n):
        S = float(spy_close.iloc[i])
        vix_val = float(vix_close.iloc[i])
        atm_sigma = vix_val / 100.0
        skew = estimate_skew(vix_val)
        otm_sigma = atm_sigma + skew  # OTM put has higher IV
        dt = dates[i]

        # --- Manage active trade ---
        if active_trade is not None:
            days_held = i - active_trade['entry_idx']
            remaining_dte = active_trade['dte'] - days_held

            # Current skew for exit check
            current_skew = estimate_skew(vix_val)

            if remaining_dte <= 0:
                # Expiration
                otm_intrinsic = max(active_trade['otm_strike'] - S, 0)
                atm_intrinsic = max(active_trade['atm_strike'] - S, 0)
                # We sold OTM put (short), bought ATM put (long)
                # P&L = credit received - (otm_intrinsic - atm_intrinsic)
                spread_intrinsic = otm_intrinsic - atm_intrinsic  # negative = max loss
                pnl = (active_trade['credit'] + spread_intrinsic) * 100 - commission * 2
                active_trade['pnl'] = pnl
                active_trade['exit_date'] = dt
                active_trade['exit_reason'] = 'expiration'
                equity += pnl
                trades.append(active_trade)
                active_trade = None

            elif current_skew < skew_exit:
                # Skew normalized — close
                T_rem = max(remaining_dte / 252, 1/252)
                otm_put_now = bsm_put(S, active_trade['otm_strike'], T_rem, r, otm_sigma)
                atm_put_now = bsm_put(S, active_trade['atm_strike'], T_rem, r, atm_sigma)
                spread_now = otm_put_now - atm_put_now  # cost to close
                pnl = (active_trade['credit'] - spread_now) * 100 - commission * 2
                active_trade['pnl'] = pnl
                active_trade['exit_date'] = dt
                active_trade['exit_reason'] = 'skew_normalized'
                equity += pnl
                trades.append(active_trade)
                active_trade = None

            else:
                # Check profit target
                T_rem = max(remaining_dte / 252, 1/252)
                otm_put_now = bsm_put(S, active_trade['otm_strike'], T_rem, r, otm_sigma)
                atm_put_now = bsm_put(S, active_trade['atm_strike'], T_rem, r, atm_sigma)
                spread_now = otm_put_now - atm_put_now
                unrealized = (active_trade['credit'] - spread_now) * 100
                max_profit = active_trade['credit'] * 100
                if unrealized >= profit_target_pct * max_profit:
                    pnl = unrealized - commission * 2
                    active_trade['pnl'] = pnl
                    active_trade['exit_date'] = dt
                    active_trade['exit_reason'] = 'profit_target'
                    equity += pnl
                    trades.append(active_trade)
                    active_trade = None

        # --- Entry logic ---
        if active_trade is None and (i - last_entry_idx) >= 21:
            if skew >= skew_entry:
                otm_K = S * (1 - otm_pct)
                atm_K = round(S)  # ATM strike
                T = dte / 252

                otm_premium = bsm_put(S, otm_K, T, r, otm_sigma)
                atm_premium = bsm_put(S, atm_K, T, r, atm_sigma)
                credit = otm_premium - atm_premium  # net credit from spread

                if credit > 0.05:
                    width = atm_K - otm_K  # spread width
                    max_loss_per_contract = (width - credit) * 100
                    risk_amount = equity * risk_per_trade_pct
                    contracts = max(1, int(risk_amount / max(max_loss_per_contract, 1)))

                    active_trade = {
                        'entry_date': dt,
                        'entry_idx': i,
                        'entry_price': S,
                        'otm_strike': otm_K,
                        'atm_strike': atm_K,
                        'credit': credit,
                        'width': width,
                        'contracts': contracts,
                        'dte': dte,
                        'iv': atm_sigma,
                        'vix': vix_val,
                        'skew': skew,
                    }
                    last_entry_idx = i

        equity_curve.append((dt, equity))

    # Close remaining at end
    if active_trade is not None:
        S = float(spy_close.iloc[-1])
        atm_sigma = float(vix_close.iloc[-1]) / 100
        skew = estimate_skew(float(vix_close.iloc[-1]))
        otm_sigma = atm_sigma + skew
        days_held = n - 1 - active_trade['entry_idx']
        remaining_dte = active_trade['dte'] - days_held
        T_rem = max(remaining_dte / 252, 1/252)
        otm_put_now = bsm_put(S, active_trade['otm_strike'], T_rem, r, otm_sigma)
        atm_put_now = bsm_put(S, active_trade['atm_strike'], T_rem, r, atm_sigma)
        spread_now = otm_put_now - atm_put_now
        pnl = (active_trade['credit'] - spread_now) * 100 - commission * 2
        active_trade['pnl'] = pnl
        active_trade['exit_date'] = dates[-1]
        active_trade['exit_reason'] = 'end_of_backtest'
        equity += pnl
        trades.append(active_trade)
        equity_curve.append((dates[-1], equity))

    eq_series = pd.Series(
        [v for _, v in equity_curve],
        index=[d for d, _ in equity_curve],
    )
    return trades, equity_curve, eq_series


print('Skew trading backtest engine defined.')

In [ ]:
# Run skew trading backtest
sk_trades, sk_eq_curve, sk_equity = run_skew_trading_backtest(spy_close, vix_close)

sk_returns = sk_equity.pct_change().dropna()
sk_sharpe = float(sk_returns.mean() / sk_returns.std() * np.sqrt(252)) if sk_returns.std() > 0 else 0
sk_peak = sk_equity.cummax()
sk_dd = ((sk_equity - sk_peak) / sk_peak * 100)
sk_max_dd = float(sk_dd.min())

sk_wins = sum(1 for t in sk_trades if t.get('pnl', 0) > 0)
sk_win_rate = sk_wins / len(sk_trades) * 100 if sk_trades else 0
sk_total_pnl = sum(t.get('pnl', 0) for t in sk_trades)
sk_pos_pnl = sum(t['pnl'] for t in sk_trades if t['pnl'] > 0)
sk_neg_pnl = abs(sum(t['pnl'] for t in sk_trades if t['pnl'] <= 0))
sk_pf = sk_pos_pnl / sk_neg_pnl if sk_neg_pnl > 0 else float('inf')

print('=== Skew Trading Strategy Results ===')
print(f'Trades:          {len(sk_trades)}')
print(f'Win rate:        {sk_win_rate:.1f}%')
print(f'Total P&L:       ${sk_total_pnl:,.0f}')
print(f'Profit factor:   {sk_pf:.2f}')
print(f'Sharpe ratio:    {sk_sharpe:.2f}')
print(f'Max drawdown:    {sk_max_dd:.2f}%')
print(f'Final equity:    ${sk_equity.iloc[-1]:,.0f}')
print(f'Annual return:   {(sk_equity.iloc[-1]/1_000_000)**(1/7) - 1:.2%}')

# Close reasons
reasons = {}
for t in sk_trades:
    r = t.get('exit_reason', 'unknown')
    reasons[r] = reasons.get(r, 0) + 1
print(f'\nClose reasons:  {reasons}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Equity curve
axes[0, 0].plot(sk_equity.index, sk_equity.values / 1e6, 'b-', linewidth=1)
axes[0, 0].set_title('Skew Trading — Equity Curve')
axes[0, 0].set_ylabel('Equity ($M)')
axes[0, 0].axhline(1.0, color='gray', linestyle='--', alpha=0.5)

# Drawdown
axes[0, 1].fill_between(sk_dd.index, sk_dd.values, 0, alpha=0.4, color='red')
axes[0, 1].set_title('Skew Trading — Drawdown')
axes[0, 1].set_ylabel('Drawdown (%)')

# Trade P&L distribution
pnls = [t['pnl'] for t in sk_trades]
axes[1, 0].hist(pnls, bins=30, edgecolor='black', alpha=0.7)
axes[1, 0].axvline(0, color='red', linestyle='--')
axes[1, 0].set_title('Skew Trading — Trade P&L Distribution')
axes[1, 0].set_xlabel('P&L ($)')

# P&L by skew at entry
skew_at_entry = [t.get('skew', 0) * 100 for t in sk_trades]
pnl_by_skew = [(t.get('skew', 0) * 100, t['pnl']) for t in sk_trades]
if pnl_by_skew:
    s_arr, p_arr = zip(*pnl_by_skew)
    colors = ['green' if p > 0 else 'red' for p in p_arr]
    axes[1, 1].scatter(s_arr, p_arr, c=colors, alpha=0.6, s=30)
    axes[1, 1].axhline(0, color='black', linestyle='-', alpha=0.3)
axes[1, 1].set_title('Skew Trading — P&L vs Skew at Entry')
axes[1, 1].set_xlabel('Put Skew (%)')
axes[1, 1].set_ylabel('P&L ($)')

plt.tight_layout()
plt.savefig('../reports/skew_trading_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to reports/skew_trading_analysis.png')

## 5. Comparison Table

Compare all three strategies side by side.

In [ ]:
# Delta Neutral baseline (from backtest JSON)
dn_sharpe = 1.30
dn_max_dd = 15.43
dn_win_rate = 78.9
dn_pf = 4.4
dn_annual_ret = 32.78
dn_trades = 19

comparison = pd.DataFrame({
    'Metric': [
        'Trades', 'Win Rate', 'Sharpe', 'Max Drawdown',
        'Profit Factor', 'Annual Return',
        'Complexity', 'Delta Hedging', 'Data Required',
    ],
    'Delta Neutral': [
        dn_trades, f'{dn_win_rate:.1f}%', f'{dn_sharpe:.2f}',
        f'{dn_max_dd:.1f}%', f'{dn_pf:.1f}', f'{dn_annual_ret:.1f}%',
        'High', 'Yes (daily)', 'VIX + Greeks',
    ],
    'Put Write': [
        len(pw_trades), f'{pw_win_rate:.1f}%', f'{pw_sharpe:.2f}',
        f'{pw_max_dd:.1f}%', f'{pw_pf:.1f}',
        f'{(pw_equity.iloc[-1]/1e6)**(1/7)-1:.1%}',
        'Low', 'No', 'VIX only',
    ],
    'Skew Trading': [
        len(sk_trades), f'{sk_win_rate:.1f}%', f'{sk_sharpe:.2f}',
        f'{sk_max_dd:.1f}%', f'{sk_pf:.1f}',
        f'{(sk_equity.iloc[-1]/1e6)**(1/7)-1:.1%}',
        'Medium', 'No', 'VIX + skew model',
    ],
})

print(comparison.to_string(index=False))

In [ ]:
# Overlay equity curves
fig, ax = plt.subplots(figsize=(14, 7))

# Normalize all to $1M start
pw_norm = pw_equity / pw_equity.iloc[0] * 1_000_000
sk_norm = sk_equity / sk_equity.iloc[0] * 1_000_000

ax.plot(pw_norm.index, pw_norm.values / 1e6, label='Put Write', linewidth=1.2)
ax.plot(sk_norm.index, sk_norm.values / 1e6, label='Skew Trading', linewidth=1.2)
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5, label='Initial Capital')

ax.set_title('VRP Strategy Comparison (2018–2024)', fontsize=14)
ax.set_ylabel('Equity ($M)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/vrp_strategy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to reports/vrp_strategy_comparison.png')

## 6. Regime Analysis — When Does Each Strategy Work Best?

In [ ]:
# Analyze put write performance by VIX regime
def regime_analysis(trades: list[dict], label: str) -> None:
    """Break down trade P&L by VIX regime."""
    regimes = {
        'Low (VIX<15)': [],
        'Normal (15-20)': [],
        'Elevated (20-30)': [],
        'High (VIX>30)': [],
    }
    for t in trades:
        vix = t.get('vix', 20)
        pnl = t.get('pnl', 0)
        if vix < 15:
            regimes['Low (VIX<15)'].append(pnl)
        elif vix < 20:
            regimes['Normal (15-20)'].append(pnl)
        elif vix < 30:
            regimes['Elevated (20-30)'].append(pnl)
        else:
            regimes['High (VIX>30)'].append(pnl)

    print(f'\n{label} — Regime Breakdown:')
    print(f'{"Regime":<22} {"Trades":>6} {"Win%":>6} {"Avg P&L":>10} {"Total P&L":>12}')
    print('-' * 60)
    for regime, pnls in regimes.items():
        if not pnls:
            print(f'{regime:<22} {0:>6}')
            continue
        n = len(pnls)
        wins = sum(1 for p in pnls if p > 0)
        avg = np.mean(pnls)
        total = sum(pnls)
        print(f'{regime:<22} {n:>6} {wins/n*100:>5.0f}% ${avg:>9,.0f} ${total:>11,.0f}')


regime_analysis(pw_trades, 'Put Write')
regime_analysis(sk_trades, 'Skew Trading')

## 7. Summary of Findings

### Which strategy is most robust?

**Delta Neutral VRP** remains the strongest candidate for live deployment because:
- It has the highest validated Sharpe (1.30 WF, CI_lower 0.33) with real VIX data.
- The gamma scalping component provides a structural edge in volatile markets.
- It passed the Deflated Sharpe Ratio test (DSR p-value < 0.10).

**Put Write** is the simplest strategy and captures the equity put skew premium
directly.  Its main risk is tail events (e.g., March 2020) where a single trade
can lose 3–5× the credit received.  It is a good *addition* to Delta Neutral
because the two strategies are partially uncorrelated (put write profits from
time decay; delta neutral profits from VRP spread).

**Skew Trading** adds a second dimension (skew timing) but the synthetic skew
model is a crude approximation.  Its value will only become clear with real
options chain data that captures actual skew dynamics.

### What additional data would confirm these findings?

1. **Real options chain data** (Polygon.io or IBKR) — actual bid/ask for OTM puts,
   real Greeks, and real IV at different strikes.
2. **CBOE SKEW index** — a direct measure of put skew to replace the synthetic
   model in the Skew Trading strategy.
3. **Intraday VIX** — to capture intraday VRP dynamics and improve timing.
4. **Options volume and open interest** — to filter for liquid strikes.

### Recommendation for next offline strategy

Implement **Put Write** as a Python module (`quantcore/strategies/options/put_write.py`)
with walk-forward validation.  It is the simplest to implement (no delta hedging),
has a clear structural rationale (equity put skew), and can be validated with the
same VIX proxy data.  If the WF Sharpe CI lower bound exceeds 0.3, it can join
the Delta Neutral strategy in the live paper trading portfolio.

The Skew Trading strategy should wait until real options data is available
(see `docs/options_data_roadmap.md`).